In [1]:
from pathlib import Path
import os
import re
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multitest import multipletests

pd.set_option("display.max_columns", 20)
sns.set_theme(style="whitegrid", context="talk")

ancovas_mean_results_path = '../../../../results/pet/paper_analysis/ancova_rois_mean/'
ancovas_cerebellum_results_path = '../../../../results/pet/paper_analysis/ancova_rois_cerebellum/'
neurocloud_results_path = '../../../../results/pet/paper_analysis/neurocloud/'
rois_to_lobe_csv = os.path.join('../../../../data/pet/', 'hammers_roi_to_lobe.csv')
rois_abbreviations_csv = os.path.join('../../../../data/pet/', 'roi_abbreviations.csv')
output_path = os.path.join('../../../../results/pet/paper_analysis/', 'consensus/')
os.makedirs(output_path, exist_ok=True)

# Load rois to lobe mapping
rois_to_lobe_df = pd.read_csv(rois_to_lobe_csv)
# Load abbreviations for ROIs
rois_abbreviations_df = pd.read_csv(rois_abbreviations_csv)


In [ ]:
# Perform consensus analysis between ceunim controls, cermep and neurocloud
csv_ceunim_controls_filename = os.path.join(ancovas_mean_results_path, 'ancova_lc_vs_ceunim_controls_results_time_to_acquisition.csv')
csv_cermep_filename = os.path.join(ancovas_mean_results_path, 'ancova_lc_vs_cermep_controls_results.csv')
csv_neurocloud_filename = os.path.join(neurocloud_results_path, 'ttest_neurocloud_long_covid_hammers_names.csv')  
# Read each csv
ceunim_controls_df = pd.read_csv(csv_ceunim_controls_filename)
cermep_df = pd.read_csv(csv_cermep_filename)
neurocloud_df = pd.read_csv(csv_neurocloud_filename)

# Now we identify:
# First, the ROIs that p_FDR<=0.05 for the three datasets
# For cerunim and cermep the fiels are: ROI,n,F_group,p_group,beta_group,mean_control,mean_lc,delta_lc_minus_control,direction,p_sex,beta_sex,p_age,beta_age,p_fdr,significant_fdr
# For neurocloud the fields are:Region,N,MEAN_Z,MEDIAN_Z,STD_Z,COHEN_D,TSTAT,PVAL_T,WSTAT,PVAL_W,Effect
ceunim_controls_significant_rois = ceunim_controls_df[ceunim_controls_df['p_fdr'] <= 0.05]['ROI']
cermep_significant_rois = cermep_df[cermep_df['p_fdr'] <= 0.05]['ROI']
neurocloud_significant_rois = neurocloud_df[neurocloud_df['PVAL_T'] <= 0.05]['Region']

consensus_rois = set(ceunim_controls_significant_rois) & set(cermep_significant_rois) & set(neurocloud_significant_rois)
# and a second condition is that the direction is the same across the three datasets, therefore mean_lc and MEAN_Z have the same sign
consensus_rois_with_same_direction = set()
for roi in consensus_rois:
    mean_lc_ceunim = ceunim_controls_df.loc[ceunim_controls_df['ROI'] == roi, 'mean_lc'].values[0]
    mean_lc_cermep = cermep_df.loc[cermep_df['ROI'] == roi, 'mean_lc'].values[0]
    mean_z_neurocloud = neurocloud_df.loc[neurocloud_df['Region'] == roi, 'MEAN_Z'].values[0]
    if (mean_lc_ceunim > 0 and mean_lc_cermep > 0 and mean_z_neurocloud > 0) or (mean_lc_ceunim < 0 and mean_lc_cermep < 0 and mean_z_neurocloud < 0):
        consensus_rois_with_same_direction.add(roi)
print("Consensus ROIs with the same direction across all three datasets:")
print(consensus_rois_with_same_direction)
print("Consensus ROIs with p_FDR <= 0.05 for all three datasets:")
print(consensus_rois)

# Now add to the list those which are significant in at least two datasets
from collections import Counter
all_significant_rois = list(ceunim_controls_significant_rois) + list(cermep_significant_rois) + list(neurocloud_significant_rois)
roi_counts = Counter(all_significant_rois)
at_least_two_datasets_rois = {roi for roi, count in roi_counts.items() if count >= 2}
print("ROIs significant in at least two datasets:")
print(at_least_two_datasets_rois)

# Map ROIs significant in at least two datasets to lobes
at_least_two_datasets_rois_lobes = rois_to_lobe_df[rois_to_lobe_df['ROI'].isin(at_least_two_datasets_rois)]
print("ROIs significant in at least two datasets with their corresponding lobes:")
print(at_least_two_datasets_rois_lobes)

# Map consensus ROIs to lobes
consensus_rois_lobes = rois_to_lobe_df[rois_to_lobe_df['ROI'].isin(consensus_rois)]
print("Consensus ROIs with their corresponding lobes:")
print(consensus_rois_lobes)

# Map consensus ROIs with the same direction to lobes
consensus_rois_with_same_direction_lobes = rois_to_lobe_df[rois_to_lobe_df['ROI'].isin(consensus_rois_with_same_direction)]
print("Consensus ROIs with the same direction across all three datasets with their corresponding lobes:")
print(consensus_rois_with_same_direction_lobes)




FileNotFoundError: [Errno 2] No such file or directory: '../../../../results/pet/paper_analysis/ancova_rois_mean/ancova_lc_vs_cermep_results.csv'